# Práctica 1 Aprendizaje Automático: Predicción de Subscripción a un Producto Bancario

**Miembros del equipo:**
    Jose Luis Mejía 1: (NIA:100472712)
    Mireya Luque 2: (NIA:100495927)

____________________________________________________________________________________________________

Declaración de uso de IA Generativa

In [1]:
### IMPORTS PARA EL NOTEBOOK

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

## Carga inicial de datos y configuración de reproducibilidad

En este bloque se fija una **semilla aleatoria** usando un NIA del equipo (**100495927**) con `np.random.seed(NIA)`.  
Esto garantiza que cualquier proceso que dependa de aleatoriedad (por ejemplo, particiones train test o validación cruzada) produzca los mismos resultados cada vez que se ejecute el notebook.

A continuación, se define la ruta del archivo `bank_09.pkl` y se carga el dataset con `pd.read_pickle`.

Finalmente, se muestra:
- La **dimensión del dataset** (número de filas y columnas) para tener una visión general.
- Las **primeras filas** con `head()` para comprobar que la carga se ha realizado correctamente y ver un ejemplo del contenido.

In [2]:
#Fijamos la semilla con un NIA del equipo
NIA = 100495927
np.random.seed(NIA)

#Cargamos los datos
ruta_archivo = 'Datos/bank_09.pkl'
df = pd.read_pickle(ruta_archivo)

#Mostramos la información
print("Dimensiones del dataset:", df.shape)
display(df.head())

Dimensiones del dataset: (11000, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


## EDA simplificado: estructura, tipos de variables y calidad de datos

En este bloque se realiza un **EDA simplificado** (Exploratory Data Analysis) para obtener una visión general del dataset y detectar posibles problemas de calidad de datos que afecten al preprocesamiento y al modelado.

### 1. Instancias y variables
Se imprime el número de **filas** (instancias) y **columnas** (variables) del dataset usando `df.shape`. Esto permite conocer el tamaño del conjunto de datos con el que se va a trabajar.

### 2. Tipos de variables
Se separan las variables en dos grandes grupos:
- **Numéricas**: columnas de tipo `int64` o `float64`.
- **Categóricas**: columnas de tipo `object`, `category` o `bool`.

Esto es importante porque cada tipo de variable suele requerir un preprocesamiento distinto (por ejemplo, escalado para numéricas y codificación para categóricas).

### 3. Valores faltantes
Se calcula cuántos valores nulos hay en cada columna con `df.isnull().sum()`.  
Después, se filtran solo aquellas columnas que realmente tengan valores faltantes (más de 0).  
Si no hay columnas con nulos, se indica explícitamente.

### 4. Columnas constantes
Se detectan columnas con **un único valor** (o sin variación) calculando el número de valores únicos con `nunique()`.  
Las columnas constantes no aportan información predictiva y, normalmente, se pueden eliminar.

### 5. Alta cardinalidad en variables categóricas
Se identifican variables categóricas con **más de 10 valores únicos**.  
Este punto es relevante porque la alta cardinalidad puede complicar técnicas como el one hot encoding, aumentando mucho la dimensión del dataset y el riesgo de sobreajuste.

### 6. Tipo de problema y balanceo de la variable objetivo
Se confirma que el problema es de **clasificación binaria**, ya que la variable objetivo es `deposit` (suscribe o no suscribe).  
Además, se calcula el porcentaje de cada clase con `value_counts(normalize=True)` para comprobar si el dataset está **balanceado o desbalanceado**, lo cual puede afectar tanto a la métrica elegida como al rendimiento de algunos modelos.

### 7. Análisis particular de `pdays`
Se analiza específicamente la variable `pdays`, que representa el número de días desde el último contacto en una campaña anterior.
Para ello:
- Se muestran estadísticos descriptivos (`describe()`).
- Se listan los valores más frecuentes (`value_counts().head()`).
- Se calcula el porcentaje de clientes con `pdays = -1`, que según la descripción indica **sin contacto previo o desconocido**.


In [3]:
print("--- 1. INSTANCIAS Y VARIABLES ---")
print(f"Número de instancias (filas): {df.shape[0]}")
print(f"Número de variables (columnas): {df.shape[1]}")

print("--- 2. TIPOS DE VARIABLES ---")
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categoricas = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"Variables numéricas: ({len(numericas)}): {numericas}")
print(f"Variables categóricas: ({len(categoricas)}): {categoricas}")

print("--- 3. VALORES FALTANTES ---")
nulos = df.isnull().sum()
nulos_reales = nulos[nulos > 0]
if len(nulos_reales) > 0:
    print(nulos_reales)
else:
    print("No hay valores faltantes en ninguna columna.\n")

print(" --- 4. COLUMNAS CONSTANTES ---")
constantes = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Columnas constantes o con un solo valor: {constantes if constantes else 'Ninguna'}\n")

print("--- 5. ALTA CARDINALIDAD EN CATEGÓRICAS (>10 valores únicos) ---")
alta_cardinalidad = [col for col in categoricas if df[col].nunique() > 10]
for col in alta_cardinalidad:
    print(f"-{col}: {df[col].nunique()} valores únicos")
if not alta_cardinalidad:
    print("No hay variables categóricas con alta cardinalidad.\n")
    
print("--- 6. PROBLEMA Y BALANCEO DE LA VARIABLE OBJETIVO ---")
#Sabemos que la variable objetivo es 'deposit'
print("Problema: Clasificación (predecir si se suscribe o no el depósito)")
balanceo = df['deposit'].value_counts(normalize=True) * 100
print("Balanceo de clases en 'deposit' (%):")
print(balanceo)

print("\n--- 7. ANÁLISIS PARTICULAR DE 'pdays' ---")
#pdays es el número de días que han pasado desde el último contacto en campaña anterior
print("Descripción de 'pdays':")
print(df['pdays'].describe())
print("\nTop 5 valores más comunes en pdays:")
print(df['pdays'].value_counts().head())
print(f"\nPorcentaje de clientes sin contacto previo (pdays = -1 o desconocido): {round((df['pdays'] == -1).mean() * 100, 2)}%")


--- 1. INSTANCIAS Y VARIABLES ---
Número de instancias (filas): 11000
Número de variables (columnas): 17
--- 2. TIPOS DE VARIABLES ---
Variables numéricas: (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Variables categóricas: (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit']
--- 3. VALORES FALTANTES ---
job         97
marital    282
dtype: int64
 --- 4. COLUMNAS CONSTANTES ---
Columnas constantes o con un solo valor: Ninguna

--- 5. ALTA CARDINALIDAD EN CATEGÓRICAS (>10 valores únicos) ---
-job: 12 valores únicos
-month: 12 valores únicos
--- 6. PROBLEMA Y BALANCEO DE LA VARIABLE OBJETIVO ---
Problema: Clasificación (predecir si se suscribe o no el depósito)
Balanceo de clases en 'deposit' (%):
deposit
no     52.545455
yes    47.454545
Name: proportion, dtype: float64

--- 7. ANÁLISIS PARTICULAR DE 'pdays' ---
Descripción de 'pdays':
count    11000.000000
mean        51.308636
std        108.78284

# Estrategia de evaluación del modelo

En esta sección definimos cómo vamos a evaluar los modelos durante toda la práctica. La idea principal es **no usar el conjunto de test para tomar decisiones**, sino reservarlo para el final. Así evitamos “hacer trampa” sin querer y obtenemos una estimación más realista del rendimiento.

Según el enunciado, usaremos dos niveles de evaluación:

## 1) Evaluación externa (outer): Holdout
Vamos a hacer una partición fija del dataset en dos partes:
- **Train (entrenamiento): 2/3 de los datos**
- **Test (prueba): 1/3 de los datos**

El conjunto **test** se guardará y **no se tocará durante el desarrollo**. Solo lo usaremos una única vez al final, cuando ya hayamos elegido el mejor enfoque, para estimar cómo funcionaría el modelo con datos nuevos (rendimiento “futuro”).

## 2) Evaluación interna (inner): validación cruzada sobre train
Para comparar modelos (por ejemplo KNN, árboles, etc.) y para ajustar hiperparámetros, trabajaremos **solo con el conjunto de entrenamiento (train)** usando **validación cruzada (cross-validation)**.

La validación cruzada consiste en dividir el train en varias partes (folds) y repetir entrenamientos y validaciones cambiando qué parte se usa para validar. Después se promedian los resultados. Esto nos permite:
- Comparar modelos de forma más estable
- Elegir hiperparámetros sin usar el test

## Métrica principal
Como el objetivo `deposit` tiene dos clases (sí/no), es un problema de **clasificación binaria**.  
Usaremos **Accuracy** como métrica principal durante la práctica.

Más adelante, cuando evaluemos el modelo final en el conjunto test, también analizaremos otras medidas como la **matriz de confusión** para entender mejor los aciertos y errores.

## División Train-Test (Holdout)

En este bloque se divide el dataset en dos conjuntos: **entrenamiento (train)** y **prueba (test)** usando una estrategia de tipo *holdout*.

La partición se hace con esta proporción:
- **2/3** de los datos para **train**
- **1/3** de los datos para **test**

El conjunto **train** se utilizará durante casi toda la práctica (comparar modelos, ajustar hiperparámetros, etc.).  
El conjunto **test** se reservará para el final, para evaluar el rendimiento del modelo definitivo con datos que no se han usado durante el desarrollo.

Además, se aplica **estratificación** para que la variable objetivo (por ejemplo `deposit`) mantenga aproximadamente el mismo porcentaje de clases en train y en test. Esto es importante para que ambos conjuntos sean representativos, especialmente si las clases están desbalanceadas.

In [ ]:
X = df.drop("deposit", axis=1)
y = df["deposit"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (7370, 16)
Test size: (3630, 16)


## Evaluación interna (inner)

Para comparar modelos y ajustar hiperparámetros vamos a trabajar únicamente con el conjunto **train** usando validación cruzada.

En concreto, usaremos **validación cruzada estratificada** (Stratified K-Fold) con **3 particiones** como se nos recomienda, para que en cada fold se mantenga aproximadamente la proporción de clases de `deposit`.  
Esto nos permite comparar alternativas y hacer HPO sin utilizar el conjunto **test**, que se reservará para la evaluación final (outer).

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=100495927
)

# Entrenamiento y Evaluación de modelos

En esta fase vamos a entrenar y comparar **distintos algoritmos** de clasificación para realizar la predicción de si un cliente suscribirá o no el depósito.

## 1) K-Nearest Neighbors (KNN): Selección del mejor escalador

El algoritmo KNN se basa en medir distancias entre los clientes. Por ello, es fundamental **escalar** las variables numéricas para que el saldo en este caso no domine sobre otras variables. Para ello probaremos el modelo KNN con sus **hiperparámetros por defecto** y evaluaremos qué técnica de escalado nos da mejor resultado. 

Usaremos un `pipeline` para aplicar `OneHotEncoder` a las variables categóricas y probaremos los tres escaladores diferentes:
- StandardScaler
- MinMaxScaler
- RobustScaler


Mediremos tanto la métrica de *Accuracy* mediante **3-Fold CrossValidation** como el tiempo de ejecución para elegir la mejor alternativa

In [ ]:
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate

#Separamos las columnas numéricas y categóricas para el preprocesador
columnas_numericas = X_train.select_dtypes(include=['int32','int64', 'float64']).columns.tolist()
columnas_categoricas = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

#Definimos los tres escaladores
escaladores = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

resultados_knn = []

print("--- EVALUACIÓN DE ESCALDORES CON KNN ---")
for nombre, scaler in escaladores.items():

    #1) Creamos el preprocesador de columnas
    preprocesador = ColumnTransformer(
        transformers=[
            ('num', scaler, columnas_numericas),
            ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas)
        ])
    
    #2) Creamos la pipeline con el modelo KNN por defecto
    pipeline_knn = Pipeline(steps=[
        ('preprocesamiento', preprocesador),
        ('modelo', KNeighborsClassifier())
    ])

    #3) Evaluamos con validación cruzada y medimos los tiempos
    start_time = time.time()
    cv_results = cross_validate(pipeline_knn, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    tiempo_total = time.time() - start_time
    accuracy_media = cv_results['test_score'].mean()

    #4) Guardamos los resultados
    resultados_knn.append({
        'Escalador': nombre,
        'Accuracy Media': round(accuracy_media, 4),
        'Tiempo Total (s)': round(tiempo_total, 4)
    })

    #print(f"-> {nombre} | Accuracy: {accuracy_media:.4f} | Tiempo: {tiempo_total:.2f} segundos")

#Mostramos los resultados en un DataFrame 
df_resultados_knn = pd.DataFrame(resultados_knn)
display(df_resultados_knn)
    


--- EVALUACIÓN DE ESCALDORES CON KNN ---


,Escalador,Accuracy Media,Tiempo Total (s)
0,StandardScaler,0.8016,3.1539
1,MinMaxScaler,0.7318,2.6967
2,RobustScaler,0.7853,2.6712


El método que ha demostrado un mejor comportamiento para este conjunto de datos es el `StandardScaler`, ya que proporciona un rendimiento predictivo significativamente superior. Aunque su tiempo de ejecución es ligeramente mayor, la **mejora en el rendimiento** justifica plenamente su elección. En definitiva, este será el escalador que utilizaremos de aquí en adelante para todos los modelos que requieran un escalado de datos.